# 11b ONNX Selective Quantization Boundary v1

11a에서 확인한 결론은 단순했다.

- 전체 그래프를 한 번에 int8로 양자화하면 로컬 속도는 크게 빨라진다.
- 하지만 lane 개수, decoder 결과, steering mode가 달라진다.
- 즉 **속도는 얻었지만 주행 의미가 깨졌다**.

그래서 11b는 다른 질문을 던진다.

> 모델 전체가 아니라, 어느 부분까지만 int8로 바꾸면 의미가 유지되는가?

이 노트북은 ONNX 그래프의 노드를 `backbone`, `neck`, `heads`로 분류하고, `heads`는 일단 FP32로 보존한다.
그 다음 backbone/neck 쪽만 선택적으로 양자화해서 `raw → decode → steering` 계약이 유지되는지 확인한다.

## 핵심 개념: selective quantization

ONNX Runtime의 `quantize_static()`은 전체 모델뿐 아니라 특정 노드만 양자화할 수 있다.

주요 인자:

- `nodes_to_quantize`: int8로 바꿀 ONNX node 이름 목록
- `nodes_to_exclude`: int8에서 제외할 node 이름 목록
- `op_types_to_quantize`: `Conv`, `Gemm`, `MatMul` 등 어떤 연산 타입을 대상으로 할지
- `quant_format`: `QDQ` 또는 `QOperator`
- `activation_type`, `weight_type`: activation/weight를 signed 또는 unsigned int8로 둘지

11b에서는 먼저 `nodes_to_quantize`를 사용한다.

이유는 명확하다. 11a에서 전체 양자화가 실패했으니, 이번에는 **머리 부분(CLRHead coordinate/classification regression)을 건드리지 않는 후보**부터 확인한다.

## 이번 노트북의 후보

후보는 보수적인 순서로 둔다.

1. `neck_only_qdq_u8s8`
   - neck Conv 2개만 양자화한다.
   - 속도 개선은 작을 수 있지만, 의미 보존 가능성이 가장 높다.

2. `backbone_stem_qdq_u8s8`, `backbone_layer1_qdq_u8s8`, ...
   - backbone을 layer 단위로 쪼개서 양자화한다.
   - 어떤 layer부터 steering 계약이 깨지는지 확인한다.

3. `backbone_layer4_block0_qdq_u8s8`, `backbone_layer4_block1_qdq_u8s8`
   - 1차 후보에서 layer4 전체가 거의 통과에 가까웠기 때문에 layer4를 block 단위로 더 쪼갠다.

4. `backbone_layer4_qdq_u8s8`
   - ResNet18의 마지막 layer4 Conv만 양자화한다.
   - backbone 일부만 건드리는 작은 후보.

5. `backbone_layer3_layer4_qdq_u8s8`
   - layer3 + layer4를 양자화한다.
   - 중간 강도 후보.

6. `backbone_all_qdq_u8s8`
   - ResNet18 backbone Conv 전체를 양자화한다.
   - heads는 FP32로 남긴다.

7. `backbone_neck_qdq_u8s8`
   - backbone + neck Conv를 양자화한다.
   - heads는 FP32로 남긴다.

8. `backbone_all_qoperator_u8s8`
   - backbone Conv 전체를 QOperator 형식으로 양자화한다.
   - Pi에서 QDQ보다 빠를 가능성이 있어 비교용으로 둔다.

선택 기준은 latency가 아니다.
먼저 `decode_count_mismatch == 0`, `steering_mode_mismatch == 0`이 통과되어야 한다.
그 다음에 latency가 의미 있다.

In [1]:
from pathlib import Path
import json
import shutil
import time
import traceback
import platform

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import onnx
import onnxruntime as ort
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

print("python platform:", platform.platform())
print("onnxruntime:", ort.__version__)

python platform: Windows-10-10.0.26200-SP0
onnxruntime: 1.23.2


In [2]:
# ----- Path configuration -----
BASE = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")

SOURCE_ONNX = BASE / "review_outputs" / "09_onnx_export_parity_v1" / "models" / "MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx"
SOURCE_ONNX_DATA = SOURCE_ONNX.with_name(SOURCE_ONNX.name + ".data")

PKG10 = BASE / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
NOTEBOOK10 = BASE / "notebooks" / "10_pi_runtime_latency_sequence_validation_v1.ipynb"

OUT_DIR = BASE / "review_outputs" / "11b_onnx_selective_quantization_boundary_v1"
MODELS_DIR = OUT_DIR / "models"
TABLES_DIR = OUT_DIR / "tables"
REPORTS_DIR = OUT_DIR / "reports"
VIS_DIR = OUT_DIR / "visuals"

for p in [OUT_DIR, MODELS_DIR, TABLES_DIR, REPORTS_DIR, VIS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert SOURCE_ONNX.exists(), SOURCE_ONNX
assert SOURCE_ONNX_DATA.exists(), SOURCE_ONNX_DATA
assert PKG10.exists(), PKG10
assert NOTEBOOK10.exists(), NOTEBOOK10

print("OUT_DIR:", OUT_DIR)
print("SOURCE_ONNX:", SOURCE_ONNX)
print("PKG10:", PKG10)

OUT_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\11b_onnx_selective_quantization_boundary_v1
SOURCE_ONNX: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\09_onnx_export_parity_v1\models\MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx
PKG10: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg


In [3]:
# ----- Experiment knobs -----
# None이면 10번 package 안의 해당 record를 전부 사용한다.
CALIB_RECORD_LIMIT = None
PARITY_RECORD_LIMIT = None
SEQUENCE_RECORD_LIMIT = 160
LATENCY_RECORD_LIMIT = 160

ORT_THREADS_LOCAL = None  # None = ORT default
ORT_WARMUP_RUNS = 5

# 11b는 의미 보존이 목적이므로 Conv부터만 본다.
OP_TYPES_CONV_ONLY = ["Conv"]

# 의미 보존 기준.
COUNT_MISMATCH_SOFT_LIMIT = 0
MODE_MISMATCH_SOFT_LIMIT = 0
MEAN_LANE_DIST_SOFT_LIMIT_PX = 2.0
MAX_STEER_DIFF_SOFT_LIMIT = 0.05

print("CALIB_RECORD_LIMIT:", CALIB_RECORD_LIMIT)
print("PARITY_RECORD_LIMIT:", PARITY_RECORD_LIMIT)
print("SEQUENCE_RECORD_LIMIT:", SEQUENCE_RECORD_LIMIT)
print("LATENCY_RECORD_LIMIT:", LATENCY_RECORD_LIMIT)

CALIB_RECORD_LIMIT: None
PARITY_RECORD_LIMIT: None
SEQUENCE_RECORD_LIMIT: 160
LATENCY_RECORD_LIMIT: 160


## 10번 runtime core 재사용

11b도 10번과 같은 preprocess, decoder, steering을 써야 한다.

여기서 중요한 건 “양자화 모델을 평가하는 후처리 코드”가 달라지면 안 된다는 점이다.
그래서 10번 노트북의 runtime core cell을 그대로 읽어서 실행한다.

즉 11b의 비교는 다음처럼 고정된다.

```text
FP32 ONNX raw      -> 10 decoder -> 10 steering
Quantized ONNX raw -> 10 decoder -> 10 steering
```

다른 것은 ONNX 모델뿐이다.

In [4]:
# ----- Minimal globals required by the 10 notebook runtime core -----
IS_PI = False
RAW_W, RAW_H = 1296, 972
CUT_HEIGHT = 445
IMG_W, IMG_H = 800, 320
NUM_PRIORS = 192
OUTPUT_DIM = 78
N_OFFSETS = 72
N_STRIPS = N_OFFSETS - 1
SAMPLE_Y = list(range(971, 444, -20))
IMAGE_CENTER_X = RAW_W / 2.0
DEFAULT_PAIR_BONUS_PX = 60.0
DEFAULT_PI_ORT_THREADS = 4
REQUIRE_SCIPY_FOR_DECODER = True
ORT_WARMUP_RUNS = int(ORT_WARMUP_RUNS)

def fs_path(path):
    path = Path(path)
    s = str(path)
    if platform.system().lower().startswith("win"):
        try:
            s = str(path.resolve())
        except Exception:
            s = str(path)
        if not s.startswith("\\\\?\\"):
            s = "\\\\?\\" + s
    return s

def exists_fs(path):
    try:
        return Path(path).exists()
    except OSError:
        return Path(fs_path(path)).exists()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def imread_bgr(path):
    data = np.fromfile(fs_path(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img

def imwrite_bgr(path, img):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(Path(path).suffix or ".jpg", img)
    if not ok:
        raise RuntimeError(f"cv2.imencode failed: {path}")
    buf.tofile(fs_path(path))

try:
    from scipy.interpolate import InterpolatedUnivariateSpline
    HAS_SCIPY = True
except Exception as exc:
    HAS_SCIPY = False
    if REQUIRE_SCIPY_FOR_DECODER:
        raise RuntimeError("SciPy is required for official-compatible Lane.to_array spline resampling.") from exc

nb10 = json.loads(NOTEBOOK10.read_text(encoding="utf-8"))
runtime_core = None
for cell in nb10["cells"]:
    if cell.get("cell_type") == "code":
        src = "".join(cell.get("source", []))
        if src.lstrip().startswith("# ----- Contracts and ONNX Runtime -----"):
            runtime_core = src
            break
assert runtime_core is not None, "10 notebook runtime core cell was not found."
exec(compile(runtime_core, "10_runtime_core", "exec"), globals())
print("Loaded runtime core from:", NOTEBOOK10)

Loaded runtime core from: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\notebooks\10_pi_runtime_latency_sequence_validation_v1.ipynb


In [5]:
# ----- Load records and contracts from the 10 package -----
decode_contract, driving_contract, _, _ = load_package_contracts(PKG10)
records = pd.read_csv(PKG10 / "t" / "records_manifest.csv")
records["image_path"] = records["image_rel"].apply(lambda rel: str(PKG10 / rel))

parity_records = records[records["role"] == "parity"].sort_values(["set", "order"]).copy()
sequence_records = records[records["role"] == "sequence"].sort_values(["order"]).copy()

if PARITY_RECORD_LIMIT is not None:
    parity_records = parity_records.head(int(PARITY_RECORD_LIMIT)).copy()
if SEQUENCE_RECORD_LIMIT is not None:
    sequence_records = sequence_records.head(int(SEQUENCE_RECORD_LIMIT)).copy()

calib_records = records.drop_duplicates("image_rel").sort_values(["set", "role", "order"]).copy()
if CALIB_RECORD_LIMIT is not None:
    calib_records = calib_records.head(int(CALIB_RECORD_LIMIT)).copy()

latency_records = sequence_records.copy()
if LATENCY_RECORD_LIMIT is not None:
    latency_records = latency_records.head(int(LATENCY_RECORD_LIMIT)).copy()

print("records total:", len(records))
print("calibration records:", len(calib_records))
print("parity records:", len(parity_records))
print("sequence records:", len(sequence_records))
print("latency records:", len(latency_records))
display(records.groupby(["set", "role"]).size().rename("count").reset_index())

records total: 356
calibration records: 356
parity records: 116
sequence records: 160
latency records: 160


,set,role,count
0,field3,parity,80
1,field3,sequence,240
2,holdout,parity,24
3,val,parity,12


## ONNX graph를 모듈별로 분류한다

Export된 ONNX node 이름은 `node_Conv_623`처럼 단순하다.
이름만 보고는 backbone인지 head인지 알기 어렵다.

하지만 node input의 weight initializer 이름은 다음처럼 모듈 경로를 보존한다.

```text
detector.backbone.model.layer4.1.conv2.weight
detector.neck.fpn_convs.0.0.weight
detector.heads.reg_layers.weight
```

따라서 11b는 node가 참조하는 weight 이름으로 group을 판정한다.

- `detector.backbone.*` -> backbone
- `detector.neck.*` -> neck
- `detector.heads.*` -> heads

이 셀의 표가 이번 선택 양자화의 근거다.

In [6]:
def inspect_onnx_nodes(model_path):
    model = onnx.load(fs_path(model_path), load_external_data=False)
    init_names = {x.name for x in model.graph.initializer}
    rows = []
    for idx, node in enumerate(model.graph.node):
        weight_inputs = [x for x in node.input if x in init_names]
        weight_text = "|".join(weight_inputs)
        if "detector.backbone." in weight_text:
            group = "backbone"
        elif "detector.neck." in weight_text:
            group = "neck"
        elif "detector.heads." in weight_text:
            group = "heads"
        else:
            group = "other"
        rows.append({
            "idx": idx,
            "name": node.name,
            "op_type": node.op_type,
            "group": group,
            "weight_inputs": weight_text,
            "outputs": "|".join(node.output),
        })
    return pd.DataFrame(rows)

node_df = inspect_onnx_nodes(SOURCE_ONNX)
node_df.to_csv(TABLES_DIR / "onnx_node_inventory.csv", index=False, encoding="utf-8-sig")

display(node_df.groupby(["group", "op_type"]).size().rename("count").reset_index().sort_values(["group", "op_type"]))
display(node_df[node_df["op_type"].isin(["Conv", "Gemm", "MatMul"])][["idx", "name", "op_type", "group", "weight_inputs"]])

,group,op_type,count
0,backbone,Conv,20
1,heads,Conv,6
2,heads,Expand,1
3,heads,Gemm,7
4,heads,LayerNormalization,1
5,neck,Conv,2
6,other,Add,13
7,other,Div,2
8,other,Expand,3
9,other,Gather,4


,idx,name,op_type,group,weight_inputs
0,0,node_Conv_623,Conv,backbone,detector.backbone.model.conv1.weight|detector....
3,3,node_Conv_625,Conv,backbone,detector.backbone.model.layer1.0.conv1.weight|...
5,5,node_Conv_627,Conv,backbone,detector.backbone.model.layer1.0.conv2.weight|...
8,8,node_Conv_629,Conv,backbone,detector.backbone.model.layer1.1.conv1.weight|...
10,10,node_Conv_631,Conv,backbone,detector.backbone.model.layer1.1.conv2.weight|...
13,13,node_Conv_633,Conv,backbone,detector.backbone.model.layer2.0.conv1.weight|...
15,15,node_Conv_635,Conv,backbone,detector.backbone.model.layer2.0.conv2.weight|...
16,16,node_Conv_637,Conv,backbone,detector.backbone.model.layer2.0.downsample.0....
19,19,node_Conv_639,Conv,backbone,detector.backbone.model.layer2.1.conv1.weight|...
21,21,node_Conv_641,Conv,backbone,detector.backbone.model.layer2.1.conv2.weight|...


In [7]:
def nodes_by(predicate):
    sub = node_df[node_df.apply(predicate, axis=1)]
    return sub["name"].dropna().astype(str).tolist()

def conv_nodes_in_group(group):
    return nodes_by(lambda r: r["group"] == group and r["op_type"] == "Conv")

def backbone_layer_nodes(layers):
    layers = tuple(layers)
    return nodes_by(
        lambda r: (
            r["group"] == "backbone"
            and r["op_type"] == "Conv"
            and any(f"detector.backbone.model.{layer}." in str(r["weight_inputs"]) for layer in layers)
        )
    )

neck_conv_nodes = conv_nodes_in_group("neck")
backbone_conv_nodes = conv_nodes_in_group("backbone")
backbone_stem_nodes = nodes_by(
    lambda r: (
        r["group"] == "backbone"
        and r["op_type"] == "Conv"
        and "detector.backbone.model.conv1." in str(r["weight_inputs"])
    )
)
backbone_layer1_nodes = backbone_layer_nodes(["layer1"])
backbone_layer2_nodes = backbone_layer_nodes(["layer2"])
backbone_layer3_nodes = backbone_layer_nodes(["layer3"])
backbone_layer4_nodes = backbone_layer_nodes(["layer4"])
backbone_layer4_block0_nodes = nodes_by(
    lambda r: (
        r["group"] == "backbone"
        and r["op_type"] == "Conv"
        and "detector.backbone.model.layer4.0." in str(r["weight_inputs"])
    )
)
backbone_layer4_block1_nodes = nodes_by(
    lambda r: (
        r["group"] == "backbone"
        and r["op_type"] == "Conv"
        and "detector.backbone.model.layer4.1." in str(r["weight_inputs"])
    )
)
backbone_layer34_nodes = backbone_layer_nodes(["layer3", "layer4"])

node_group_summary = {
    "neck_conv": neck_conv_nodes,
    "backbone_stem_conv": backbone_stem_nodes,
    "backbone_layer1_conv": backbone_layer1_nodes,
    "backbone_layer2_conv": backbone_layer2_nodes,
    "backbone_layer3_conv": backbone_layer3_nodes,
    "backbone_layer4_block0_conv": backbone_layer4_block0_nodes,
    "backbone_layer4_block1_conv": backbone_layer4_block1_nodes,
    "backbone_layer4_conv": backbone_layer4_nodes,
    "backbone_layer3_layer4_conv": backbone_layer34_nodes,
    "backbone_all_conv": backbone_conv_nodes,
    "backbone_neck_conv": backbone_conv_nodes + neck_conv_nodes,
}

for name, nodes in node_group_summary.items():
    print(f"{name}: {len(nodes)} nodes")
    print(nodes)

neck_conv: 2 nodes
['node_conv2d_20', 'node_conv2d_21']
backbone_stem_conv: 1 nodes
['node_Conv_623']
backbone_layer1_conv: 4 nodes
['node_Conv_625', 'node_Conv_627', 'node_Conv_629', 'node_Conv_631']
backbone_layer2_conv: 5 nodes
['node_Conv_633', 'node_Conv_635', 'node_Conv_637', 'node_Conv_639', 'node_Conv_641']
backbone_layer3_conv: 5 nodes
['node_Conv_643', 'node_Conv_645', 'node_Conv_647', 'node_Conv_649', 'node_Conv_651']
backbone_layer4_block0_conv: 3 nodes
['node_Conv_653', 'node_Conv_655', 'node_Conv_657']
backbone_layer4_block1_conv: 2 nodes
['node_Conv_659', 'node_Conv_661']
backbone_layer4_conv: 5 nodes
['node_Conv_653', 'node_Conv_655', 'node_Conv_657', 'node_Conv_659', 'node_Conv_661']
backbone_layer3_layer4_conv: 10 nodes
['node_Conv_643', 'node_Conv_645', 'node_Conv_647', 'node_Conv_649', 'node_Conv_651', 'node_Conv_653', 'node_Conv_655', 'node_Conv_657', 'node_Conv_659', 'node_Conv_661']
backbone_all_conv: 20 nodes
['node_Conv_623', 'node_Conv_625', 'node_Conv_627', '

## Calibration reader

Static quantization은 activation range를 알아야 한다.
그래서 calibration set 이미지를 모델에 한번 흘려보내고, 각 activation의 min/max 범위를 측정한다.

여기서 중요한 점:

- GT/label은 필요 없다.
- 학습을 다시 하는 것도 아니다.
- 이미지만 필요하다.
- 10번 package의 val/field3/holdout 이미지를 그대로 사용한다.

즉 calibration set은 “양자화 스케일을 정하기 위한 대표 입력 이미지 묶음”이다.

In [8]:
class ImageCalibrationDataReader(CalibrationDataReader):
    def __init__(self, image_paths, input_name):
        self.image_paths = list(image_paths)
        self.input_name = input_name
        self._iter = None

    def get_next(self):
        if self._iter is None:
            self._iter = iter(self.image_paths)
        try:
            p = next(self._iter)
        except StopIteration:
            return None
        bgr = imread_bgr(p)
        return {self.input_name: preprocess_bgr_for_model(bgr)}

    def rewind(self):
        self._iter = None

def copy2_ensuring_parent(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    return shutil.copy2(src, dst)

def prepare_source_model():
    local_onnx = MODELS_DIR / SOURCE_ONNX.name
    local_data = MODELS_DIR / SOURCE_ONNX_DATA.name
    copy2_ensuring_parent(SOURCE_ONNX, local_onnx)
    copy2_ensuring_parent(SOURCE_ONNX_DATA, local_data)
    return local_onnx

def make_session_from_model(model_path, intra_op_num_threads=ORT_THREADS_LOCAL):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    if intra_op_num_threads is not None:
        so.intra_op_num_threads = int(intra_op_num_threads)
    session = ort.InferenceSession(fs_path(model_path), sess_options=so, providers=["CPUExecutionProvider"])
    return session, session.get_inputs()[0].name, session.get_outputs()[0].name

def warmup_session(session, input_name, image_paths, runs=ORT_WARMUP_RUNS):
    if not image_paths:
        return
    bgr = imread_bgr(image_paths[0])
    inp = preprocess_bgr_for_model(bgr)
    for _ in range(int(runs)):
        session.run(None, {input_name: inp})

def run_raw_from_session(session, input_name, output_name, bgr):
    inp = preprocess_bgr_for_model(bgr)
    out = session.run([output_name], {input_name: inp})[0]
    assert out.shape == (1, NUM_PRIORS, OUTPUT_DIM), out.shape
    return out[0].astype(np.float32)

def file_mb(path):
    return Path(path).stat().st_size / (1024 * 1024)

## 후보 생성

여기서 실제로 selective quantization을 수행한다.

공통 설정:

- activation: `QUInt8`
- weight: `QInt8`
- calibration: `MinMax`
- 대상 op: `Conv`
- head 관련 Conv/Gemm/MatMul은 기본적으로 FP32 유지

`U8S8`를 기본으로 둔 이유는 11a에서 로컬 latency가 가장 좋았던 조합이 `static_qdq_u8s8`였기 때문이다.
이번에는 그 조합을 전체 모델이 아니라 일부 node에만 적용한다.

In [9]:
def candidate_specs_from_nodes():
    return [
        {
            "name": "neck_only_qdq_u8s8",
            "format": "QDQ",
            "nodes": neck_conv_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Only the 2 neck Conv nodes are quantized. Safest but likely small speedup.",
        },
        {
            "name": "backbone_stem_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_stem_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Only the first ResNet stem Conv is quantized.",
        },
        {
            "name": "backbone_layer1_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_layer1_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Only ResNet layer1 Conv nodes are quantized.",
        },
        {
            "name": "backbone_layer2_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_layer2_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Only ResNet layer2 Conv nodes are quantized.",
        },
        {
            "name": "backbone_layer3_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_layer3_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Only ResNet layer3 Conv nodes are quantized.",
        },
        {
            "name": "backbone_layer4_block0_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_layer4_block0_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Only ResNet layer4 block0 Conv nodes are quantized.",
        },
        {
            "name": "backbone_layer4_block1_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_layer4_block1_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Only ResNet layer4 block1 Conv nodes are quantized.",
        },
        {
            "name": "backbone_layer4_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_layer4_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Only ResNet layer4 Conv nodes are quantized.",
        },
        {
            "name": "backbone_layer3_layer4_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_layer34_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "ResNet layer3+layer4 Conv nodes are quantized.",
        },
        {
            "name": "backbone_all_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_conv_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "All ResNet backbone Conv nodes are quantized; neck and heads remain FP32.",
        },
        {
            "name": "backbone_neck_qdq_u8s8",
            "format": "QDQ",
            "nodes": backbone_conv_nodes + neck_conv_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "Backbone + neck Conv nodes are quantized; CLRHead remains FP32.",
        },
        {
            "name": "backbone_all_qoperator_u8s8",
            "format": "QOperator",
            "nodes": backbone_conv_nodes,
            "op_types": OP_TYPES_CONV_ONLY,
            "description": "All backbone Conv nodes are quantized in QOperator form for Pi speed comparison.",
        },
    ]

def generate_quantization_candidates():
    source_model = prepare_source_model()
    probe_session, input_name, _ = make_session_from_model(source_model)
    image_paths = calib_records["image_path"].tolist()
    specs = candidate_specs_from_nodes()

    rows = []
    for spec in specs:
        name = spec["name"]
        out = MODELS_DIR / f"{name}.onnx"
        t0 = time.perf_counter()
        status = "ok"
        error = ""
        try:
            assert spec["nodes"], f"{name} has no nodes_to_quantize."
            if out.exists():
                out.unlink()
            reader = ImageCalibrationDataReader(image_paths, input_name)
            quantize_static(
                model_input=source_model,
                model_output=out,
                calibration_data_reader=reader,
                quant_format=QuantFormat.QDQ if spec["format"] == "QDQ" else QuantFormat.QOperator,
                op_types_to_quantize=spec["op_types"],
                per_channel=False,
                activation_type=QuantType.QUInt8,
                weight_type=QuantType.QInt8,
                nodes_to_quantize=spec["nodes"],
                calibrate_method=CalibrationMethod.MinMax,
                use_external_data_format=False,
            )
            assert out.exists(), out
        except Exception as exc:
            status = "error"
            error = "".join(traceback.format_exception_only(type(exc), exc)).strip()

        rows.append({
            "name": name,
            "format": spec["format"],
            "path": str(out),
            "node_count": len(spec["nodes"]),
            "nodes_to_quantize": "|".join(spec["nodes"]),
            "description": spec["description"],
            "status": status,
            "error": error,
            "create_sec": time.perf_counter() - t0,
            "size_mb": file_mb(out) if out.exists() else np.nan,
        })
        print(name, status, error if error else f"{rows[-1]['size_mb']:.2f} MB, nodes={len(spec['nodes'])}")

    df = pd.DataFrame(rows)
    df.to_csv(TABLES_DIR / "candidate_generation.csv", index=False, encoding="utf-8-sig")
    return df

candidate_generation = generate_quantization_candidates()
display(candidate_generation[["name", "format", "node_count", "status", "size_mb", "create_sec", "description", "error"]])

neck_only_qdq_u8s8 ok 44.09 MB, nodes=2


backbone_stem_qdq_u8s8 ok 44.26 MB, nodes=1


backbone_layer1_qdq_u8s8 ok 43.87 MB, nodes=4


backbone_layer2_qdq_u8s8 ok 42.79 MB, nodes=5


backbone_layer3_qdq_u8s8 ok 38.29 MB, nodes=5


...

backbone_layer3_layer4_qdq_u8s8 ok 14.30 MB, nodes=10


backbone_all_qdq_u8s8 ok 12.37 MB, nodes=20


backbone_neck_qdq_u8s8 ok 12.17 MB, nodes=22


backbone_all_qoperator_u8s8 ok 12.28 MB, nodes=20


,name,format,node_count,status,size_mb,create_sec,description,error
0,neck_only_qdq_u8s8,QDQ,2,ok,44.087223,49.603323,Only the 2 neck Conv nodes are quantized. Safe...,
1,backbone_stem_qdq_u8s8,QDQ,1,ok,44.258323,54.605298,Only the first ResNet stem Conv is quantized.,
2,backbone_layer1_qdq_u8s8,QDQ,4,ok,43.868484,62.402526,Only ResNet layer1 Conv nodes are quantized.,
3,backbone_layer2_qdq_u8s8,QDQ,5,ok,42.791742,49.410057,Only ResNet layer2 Conv nodes are quantized.,
4,backbone_layer3_qdq_u8s8,QDQ,5,ok,38.291761,49.094830,Only ResNet layer3 Conv nodes are quantized.,
5,backbone_layer4_block0_qdq_u8s8,QDQ,3,ok,33.788498,49.561190,Only ResNet layer4 block0 Conv nodes are quant...,
6,backbone_layer4_block1_qdq_u8s8,QDQ,2,ok,30.787096,49.272557,Only ResNet layer4 block1 Conv nodes are quant...,
7,backbone_layer4_qdq_u8s8,QDQ,5,ok,20.291786,49.247982,Only ResNet layer4 Conv nodes are quantized.,
8,backbone_layer3_layer4_qdq_u8s8,QDQ,10,ok,14.299740,48.876154,ResNet layer3+layer4 Conv nodes are quantized.,
9,backbone_all_qdq_u8s8,QDQ,20,ok,12.366865,48.250220,All ResNet backbone Conv nodes are quantized; ...,


## 평가 방식

11b의 평가는 11a와 같은 3층 구조다.

1. Raw parity
   - FP32 ONNX raw와 quantized ONNX raw의 element-wise 차이를 본다.

2. Decode parity
   - 07/10 decoder를 적용한 lane 개수와 lane 위치가 같은지 본다.
   - 가장 중요한 값은 `decode_count_mismatch`다.

3. Steering parity
   - 08/10 steering postprocess를 sequence로 돌렸을 때 mode와 steer가 같은지 본다.
   - 가장 중요한 값은 `steering_mode_mismatch`다.

raw diff가 조금 있어도 lane/steering이 같으면 쓸 수 있다.
반대로 raw diff가 작아도 lane count나 steering mode가 바뀌면 주행 계약이 깨진 것이다.

In [10]:
def lane_pair_distance(lane_a, lane_b):
    pa = np.asarray(lane_a["points"], dtype=np.float32)
    pb = np.asarray(lane_b["points"], dtype=np.float32)
    if len(pa) == 0 or len(pb) == 0:
        return np.nan
    dists = []
    for x, y in pa:
        j = int(np.argmin(np.abs(pb[:, 1] - y)))
        if abs(float(pb[j, 1] - y)) <= 1.0:
            dists.append(abs(float(pb[j, 0] - x)))
    return float(np.mean(dists)) if dists else np.nan

def compare_lane_sets(base_lanes, cand_lanes):
    count_mismatch = int(len(base_lanes) != len(cand_lanes))
    pair_dists = []
    for a, b in zip(base_lanes, cand_lanes):
        d = lane_pair_distance(a, b)
        if np.isfinite(d):
            pair_dists.append(d)
    return {
        "base_count": len(base_lanes),
        "cand_count": len(cand_lanes),
        "count_mismatch": count_mismatch,
        "mean_pair_dist_px": float(np.mean(pair_dists)) if pair_dists else np.nan,
        "max_pair_dist_px": float(np.max(pair_dists)) if pair_dists else np.nan,
    }

def evaluate_candidate_model(name, model_path, fp32_session_bundle):
    fp32_session, fp32_input, fp32_output = fp32_session_bundle
    out_rows_raw = []
    out_rows_decode = []
    out_rows_latency = []
    out_rows_steer = []

    try:
        session, input_name, output_name = make_session_from_model(model_path)
        image_paths = latency_records["image_path"].tolist()
        warmup_session(session, input_name, image_paths)
    except Exception as exc:
        return {
            "name": name,
            "status": "runtime_error",
            "error": "".join(traceback.format_exception_only(type(exc), exc)).strip(),
            "raw_rows": pd.DataFrame(),
            "decode_rows": pd.DataFrame(),
            "steer_rows": pd.DataFrame(),
            "latency_rows": pd.DataFrame(),
        }

    for _, rec in tqdm(parity_records.iterrows(), total=len(parity_records), desc=f"{name} parity"):
        key = rec["key"]
        bgr = imread_bgr(rec["image_path"])
        fp32_raw = run_raw_from_session(fp32_session, fp32_input, fp32_output, bgr)
        cand_raw = run_raw_from_session(session, input_name, output_name, bgr)
        diff = np.abs(fp32_raw - cand_raw)
        fp32_lanes = decode_raw_to_lanes(fp32_raw, decode_contract)
        cand_lanes = decode_raw_to_lanes(cand_raw, decode_contract)
        lane_cmp = compare_lane_sets(fp32_lanes, cand_lanes)

        out_rows_raw.append({
            "candidate": name,
            "key": key,
            "set": rec["set"],
            "max_abs_diff": float(diff.max()),
            "mean_abs_diff": float(diff.mean()),
            "p99_abs_diff": float(np.quantile(diff, 0.99)),
        })
        out_rows_decode.append({
            "candidate": name,
            "key": key,
            "set": rec["set"],
            **lane_cmp,
        })

    base_mem = init_drive_memory()
    cand_mem = init_drive_memory()
    for _, rec in tqdm(sequence_records.iterrows(), total=len(sequence_records), desc=f"{name} steering"):
        key = rec["key"]
        bgr = imread_bgr(rec["image_path"])
        fp32_raw = run_raw_from_session(fp32_session, fp32_input, fp32_output, bgr)
        cand_raw = run_raw_from_session(session, input_name, output_name, bgr)
        fp32_lanes = decode_raw_to_lanes(fp32_raw, decode_contract)
        cand_lanes = decode_raw_to_lanes(cand_raw, decode_contract)
        base_row = update_drive(fp32_lanes, base_mem, driving_contract)
        cand_row = update_drive(cand_lanes, cand_mem, driving_contract)
        out_rows_steer.append({
            "candidate": name,
            "key": key,
            "order": int(rec["order"]),
            "base_mode": base_row["effective_mode"],
            "cand_mode": cand_row["effective_mode"],
            "mode_mismatch": int(base_row["effective_mode"] != cand_row["effective_mode"]),
            "base_steer": float(base_row["steer_norm"]),
            "cand_steer": float(cand_row["steer_norm"]),
            "steer_abs_diff": abs(float(base_row["steer_norm"]) - float(cand_row["steer_norm"])),
            "center_abs_diff": abs(float(base_row["smoothed_center_x"]) - float(cand_row["smoothed_center_x"])),
            "heading_abs_diff": abs(float(base_row["smoothed_heading"]) - float(cand_row["smoothed_heading"])),
        })

    for _, rec in tqdm(latency_records.iterrows(), total=len(latency_records), desc=f"{name} latency"):
        bgr = imread_bgr(rec["image_path"])
        t0 = time.perf_counter()
        inp = preprocess_bgr_for_model(bgr)
        t1 = time.perf_counter()
        raw = session.run([output_name], {input_name: inp})[0][0]
        t2 = time.perf_counter()
        lanes = decode_raw_to_lanes(raw.astype(np.float32), decode_contract)
        t3 = time.perf_counter()
        out_rows_latency.append({
            "candidate": name,
            "key": rec["key"],
            "preprocess_ms": (t1 - t0) * 1000,
            "inference_ms": (t2 - t1) * 1000,
            "decode_ms": (t3 - t2) * 1000,
            "pipeline_ms": (t3 - t0) * 1000,
            "lane_count": len(lanes),
        })

    return {
        "name": name,
        "status": "ok",
        "error": "",
        "raw_rows": pd.DataFrame(out_rows_raw),
        "decode_rows": pd.DataFrame(out_rows_decode),
        "steer_rows": pd.DataFrame(out_rows_steer),
        "latency_rows": pd.DataFrame(out_rows_latency),
    }

def measure_fp32_latency(fp32_session_bundle):
    session, input_name, output_name = fp32_session_bundle
    rows = []
    warmup_session(session, input_name, latency_records["image_path"].tolist())
    for _, rec in tqdm(latency_records.iterrows(), total=len(latency_records), desc="fp32 latency"):
        bgr = imread_bgr(rec["image_path"])
        t0 = time.perf_counter()
        inp = preprocess_bgr_for_model(bgr)
        t1 = time.perf_counter()
        raw = session.run([output_name], {input_name: inp})[0][0]
        t2 = time.perf_counter()
        lanes = decode_raw_to_lanes(raw.astype(np.float32), decode_contract)
        t3 = time.perf_counter()
        rows.append({
            "candidate": "fp32_baseline",
            "key": rec["key"],
            "preprocess_ms": (t1 - t0) * 1000,
            "inference_ms": (t2 - t1) * 1000,
            "decode_ms": (t3 - t2) * 1000,
            "pipeline_ms": (t3 - t0) * 1000,
            "lane_count": len(lanes),
        })
    return pd.DataFrame(rows)

In [11]:
# ----- Evaluate all generated candidates -----
source_model = MODELS_DIR / SOURCE_ONNX.name
fp32_session_bundle = make_session_from_model(source_model)
warmup_session(fp32_session_bundle[0], fp32_session_bundle[1], latency_records["image_path"].tolist())

fp32_latency_df = measure_fp32_latency(fp32_session_bundle)

eval_results = []
raw_all, decode_all, steer_all, latency_all = [], [], [], [fp32_latency_df]

ok_candidates = candidate_generation[candidate_generation["status"] == "ok"].copy()
for _, row in ok_candidates.iterrows():
    result = evaluate_candidate_model(row["name"], Path(row["path"]), fp32_session_bundle)
    eval_results.append({"name": result["name"], "status": result["status"], "error": result["error"]})
    if not result["raw_rows"].empty:
        raw_all.append(result["raw_rows"])
    if not result["decode_rows"].empty:
        decode_all.append(result["decode_rows"])
    if not result["steer_rows"].empty:
        steer_all.append(result["steer_rows"])
    if not result["latency_rows"].empty:
        latency_all.append(result["latency_rows"])

raw_df = pd.concat(raw_all, ignore_index=True) if raw_all else pd.DataFrame()
decode_df = pd.concat(decode_all, ignore_index=True) if decode_all else pd.DataFrame()
steer_df = pd.concat(steer_all, ignore_index=True) if steer_all else pd.DataFrame()
latency_df = pd.concat(latency_all, ignore_index=True) if latency_all else pd.DataFrame()

raw_df.to_csv(TABLES_DIR / "candidate_raw_parity.csv", index=False, encoding="utf-8-sig")
decode_df.to_csv(TABLES_DIR / "candidate_decode_parity.csv", index=False, encoding="utf-8-sig")
steer_df.to_csv(TABLES_DIR / "candidate_steering_parity.csv", index=False, encoding="utf-8-sig")
latency_df.to_csv(TABLES_DIR / "candidate_latency.csv", index=False, encoding="utf-8-sig")

pd.DataFrame(eval_results).to_csv(TABLES_DIR / "candidate_eval_status.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame(eval_results))

fp32 latency:   0%|          | 0/160 [00:00<?, ?it/s]

neck_only_qdq_u8s8 parity:   0%|          | 0/116 [00:00<?, ?it/s]

neck_only_qdq_u8s8 steering:   0%|          | 0/160 [00:00<?, ?it/s]

neck_only_qdq_u8s8 latency:   0%|          | 0/160 [00:00<?, ?it/s]

backbone_stem_qdq_u8s8 parity:   0%|          | 0/116 [00:00<?, ?it/s]

...

backbone_neck_qdq_u8s8 latency:   0%|          | 0/160 [00:00<?, ?it/s]

backbone_all_qoperator_u8s8 parity:   0%|          | 0/116 [00:00<?, ?it/s]

backbone_all_qoperator_u8s8 steering:   0%|          | 0/160 [00:00<?, ?it/s]

backbone_all_qoperator_u8s8 latency:   0%|          | 0/160 [00:00<?, ?it/s]

,name,status,error
0,neck_only_qdq_u8s8,ok,
1,backbone_stem_qdq_u8s8,ok,
2,backbone_layer1_qdq_u8s8,ok,
3,backbone_layer2_qdq_u8s8,ok,
4,backbone_layer3_qdq_u8s8,ok,
5,backbone_layer4_block0_qdq_u8s8,ok,
6,backbone_layer4_block1_qdq_u8s8,ok,
7,backbone_layer4_qdq_u8s8,ok,
8,backbone_layer3_layer4_qdq_u8s8,ok,
9,backbone_all_qdq_u8s8,ok,


In [12]:
def q95(s):
    return float(s.quantile(0.95)) if len(s) else np.nan

fp32_mean_pipeline = float(fp32_latency_df["pipeline_ms"].mean())

summary_rows = []
for _, gen in candidate_generation.iterrows():
    name = gen["name"]
    status_rows = [r for r in eval_results if r["name"] == name]
    eval_status = status_rows[0]["status"] if status_rows else ("not_run" if gen["status"] == "ok" else "not_generated")
    eval_error = status_rows[0]["error"] if status_rows else gen.get("error", "")
    raw_sub = raw_df[raw_df["candidate"] == name] if not raw_df.empty else pd.DataFrame()
    dec_sub = decode_df[decode_df["candidate"] == name] if not decode_df.empty else pd.DataFrame()
    ste_sub = steer_df[steer_df["candidate"] == name] if not steer_df.empty else pd.DataFrame()
    lat_sub = latency_df[latency_df["candidate"] == name] if not latency_df.empty else pd.DataFrame()

    decode_count_mismatch = int(dec_sub["count_mismatch"].sum()) if not dec_sub.empty else np.nan
    steering_mode_mismatch = int(ste_sub["mode_mismatch"].sum()) if not ste_sub.empty else np.nan
    mean_lane_dist = float(dec_sub["mean_pair_dist_px"].dropna().mean()) if not dec_sub.empty else np.nan
    max_lane_dist = float(dec_sub["max_pair_dist_px"].dropna().max()) if not dec_sub.empty else np.nan
    max_steer_diff = float(ste_sub["steer_abs_diff"].max()) if not ste_sub.empty else np.nan
    mean_pipeline = float(lat_sub["pipeline_ms"].mean()) if not lat_sub.empty else np.nan
    fps_mean = float(1000.0 / mean_pipeline) if np.isfinite(mean_pipeline) and mean_pipeline > 0 else np.nan
    speedup = float(fp32_mean_pipeline / mean_pipeline) if np.isfinite(mean_pipeline) and mean_pipeline > 0 else np.nan

    meaning_pass = (
        eval_status == "ok"
        and decode_count_mismatch <= COUNT_MISMATCH_SOFT_LIMIT
        and steering_mode_mismatch <= MODE_MISMATCH_SOFT_LIMIT
        and (not np.isfinite(mean_lane_dist) or mean_lane_dist <= MEAN_LANE_DIST_SOFT_LIMIT_PX)
        and (not np.isfinite(max_steer_diff) or max_steer_diff <= MAX_STEER_DIFF_SOFT_LIMIT)
    )

    summary_rows.append({
        "candidate": name,
        "format": gen["format"],
        "node_count": int(gen["node_count"]),
        "generation_status": gen["status"],
        "eval_status": eval_status,
        "meaning_pass": bool(meaning_pass),
        "size_mb": gen["size_mb"],
        "raw_max_abs_diff": float(raw_sub["max_abs_diff"].max()) if not raw_sub.empty else np.nan,
        "raw_mean_abs_diff_max": float(raw_sub["mean_abs_diff"].max()) if not raw_sub.empty else np.nan,
        "decode_count_mismatch": decode_count_mismatch,
        "decode_mean_pair_dist_px": mean_lane_dist,
        "decode_max_pair_dist_px": max_lane_dist,
        "steering_mode_mismatch": steering_mode_mismatch,
        "steering_max_abs_diff": max_steer_diff,
        "latency_pipeline_mean_ms": mean_pipeline,
        "latency_pipeline_p95_ms": q95(lat_sub["pipeline_ms"]) if not lat_sub.empty else np.nan,
        "latency_inference_mean_ms": float(lat_sub["inference_ms"].mean()) if not lat_sub.empty else np.nan,
        "fps_mean": fps_mean,
        "speedup_vs_fp32": speedup,
        "error": eval_error if eval_error else gen.get("error", ""),
        "path": gen["path"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TABLES_DIR / "candidate_eval_summary.csv", index=False, encoding="utf-8-sig")

fp32_summary = {
    "latency_pipeline_mean_ms": fp32_mean_pipeline,
    "latency_pipeline_p95_ms": q95(fp32_latency_df["pipeline_ms"]),
    "latency_inference_mean_ms": float(fp32_latency_df["inference_ms"].mean()),
    "fps_mean": float(1000.0 / fp32_mean_pipeline),
}
write_json(REPORTS_DIR / "fp32_latency_baseline.json", fp32_summary)

display(pd.DataFrame([{"candidate": "fp32_baseline", **fp32_summary}]))
display(summary_df.sort_values(["meaning_pass", "latency_pipeline_mean_ms"], ascending=[False, True]))

,candidate,latency_pipeline_mean_ms,latency_pipeline_p95_ms,latency_inference_mean_ms,fps_mean
0,fp32_baseline,73.997585,140.43661,70.224046,13.513955


,candidate,format,node_count,generation_status,eval_status,meaning_pass,size_mb,raw_max_abs_diff,raw_mean_abs_diff_max,decode_count_mismatch,decode_mean_pair_dist_px,decode_max_pair_dist_px,steering_mode_mismatch,steering_max_abs_diff,latency_pipeline_mean_ms,latency_pipeline_p95_ms,latency_inference_mean_ms,fps_mean,speedup_vs_fp32,error,path
10,backbone_neck_qdq_u8s8,QDQ,22,ok,ok,False,12.170280,1249.611450,2.539623,6,13.748052,571.091263,3,0.124643,44.941492,71.257145,40.517146,22.251152,1.646532,,~\02_Projects\University\26-1_Em...
9,backbone_all_qdq_u8s8,QDQ,20,ok,ok,False,12.366865,1325.709473,2.816716,4,8.762911,554.926031,3,0.124618,49.891264,84.579955,45.384346,20.043589,1.483177,,~\02_Projects\University\26-1_Em...
11,backbone_all_qoperator_u8s8,QOperator,20,ok,ok,False,12.284564,867.320984,1.766627,4,3.635013,250.583435,3,0.121574,54.870428,99.921100,50.590437,18.224753,1.348588,,~\02_Projects\University\26-1_Em...
8,backbone_layer3_layer4_qdq_u8s8,QDQ,10,ok,ok,False,14.299740,1212.614746,2.460646,0,1.146283,55.782462,4,0.162620,60.299966,97.888735,56.237329,16.583757,1.227158,,~\02_Projects\University\26-1_Em...
7,backbone_layer4_qdq_u8s8,QDQ,5,ok,ok,False,20.291786,1673.595703,3.392662,0,0.604530,16.498514,2,0.050746,66.299694,145.423830,61.378974,15.083026,1.116107,,~\02_Projects\University\26-1_Em...
4,backbone_layer3_qdq_u8s8,QDQ,5,ok,ok,False,38.291761,1667.040771,3.378855,1,0.901448,55.737255,4,0.161900,66.335665,106.863350,61.973850,15.074847,1.115502,,~\02_Projects\University\26-1_Em...
0,neck_only_qdq_u8s8,QDQ,2,ok,ok,False,44.087223,733.207703,1.487441,3,1.123054,51.531688,2,0.023425,69.742786,109.625545,66.080466,14.338401,1.061007,,~\02_Projects\University\26-1_Em...
6,backbone_layer4_block1_qdq_u8s8,QDQ,2,ok,ok,False,30.787096,3107.884521,6.297289,0,0.332307,2.847208,5,0.162699,75.599432,173.435055,70.820021,13.227613,0.978811,,~\02_Projects\University\26-1_Em...
2,backbone_layer1_qdq_u8s8,QDQ,4,ok,ok,False,43.868484,2473.233887,5.901047,1,1.205705,36.956183,1,0.052385,77.618022,125.711530,73.333755,12.883606,0.953356,,~\02_Projects\University\26-1_Em...
3,backbone_layer2_qdq_u8s8,QDQ,5,ok,ok,False,42.791742,1525.080078,3.091845,1,0.316027,3.453301,6,0.140546,78.268796,151.770155,73.069127,12.776484,0.945429,,~\02_Projects\University\26-1_Em...


## 선택 규칙

선택은 두 단계다.

1. 먼저 의미 보존을 통과해야 한다.
   - lane 개수 mismatch 0
   - steering mode mismatch 0
   - steering 차이가 작을 것

2. 통과 후보 중 가장 빠른 모델을 고른다.

여기서 통과 후보가 없다면 양자화를 바로 Pi로 보내지 않는다.
그 경우 다음 단계는 더 작은 범위, 예를 들어 `neck_only`, `layer4` 내부 일부 node만 양자화하는 11c가 된다.

In [13]:
valid = summary_df[summary_df["meaning_pass"] == True].copy()
if len(valid):
    selected = valid.sort_values("latency_pipeline_mean_ms").iloc[0].to_dict()
else:
    selected = None

calibration_sets = {
    f"{set_name}/{role_name}": int(count)
    for (set_name, role_name), count in calib_records.groupby(["set", "role"]).size().items()
}

report = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook": "11b_onnx_selective_quantization_boundary_v1.ipynb",
    "purpose": "Find the widest safe quantization boundary while preserving 07 decoder and 08/10 steering contracts.",
    "source_onnx": str(SOURCE_ONNX),
    "source_package": str(PKG10),
    "fp32_latency_baseline": fp32_summary,
    "calibration": {
        "records": int(len(calib_records)),
        "requires_gt": False,
        "sets": calibration_sets,
        "note": "Calibration uses only images to measure activation ranges; labels/GT are not used.",
    },
    "node_group_summary": {
        k: {"count": len(v), "nodes": v}
        for k, v in node_group_summary.items()
    },
    "candidate_generation": candidate_generation.to_dict(orient="records"),
    "summary": summary_df.to_dict(orient="records"),
    "selected_for_pi_validation": selected,
    "interpretation": (
        "If selected_for_pi_validation is null, selective quantization still changed the driving contract. "
        "Do not deploy a quantized model before a smaller-boundary experiment passes."
    ),
    "next": [
        "If a candidate passed, create 12_quantized_pi_runtime_validation_v1.ipynb using that ONNX.",
        "If no candidate passed, reduce the boundary further: individual layer4 blocks, neck-only, or weight-only experiments.",
        "Final acceptance must be based on Pi latency + local/Pi parity, not local latency alone.",
    ],
}
write_json(REPORTS_DIR / "selective_quantization_boundary_report.json", report)

if selected:
    write_json(REPORTS_DIR / "selected_candidate_for_pi.json", selected)
    print("Selected candidate for Pi validation:")
    print(json.dumps(selected, indent=2, ensure_ascii=False))
else:
    print("No selective candidate passed the meaning-preservation rules.")

print("report:", REPORTS_DIR / "selective_quantization_boundary_report.json")

No selective candidate passed the meaning-preservation rules.
report: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\11b_onnx_selective_quantization_boundary_v1\reports\selective_quantization_boundary_report.json


In [14]:
def draw_lane_compare_sheet(selected_candidate, max_frames=12):
    if selected_candidate is None:
        print("No selected candidate; skip visual sheet.")
        return None

    model_path = Path(selected_candidate["path"])
    fp32_session, fp32_input, fp32_output = make_session_from_model(MODELS_DIR / SOURCE_ONNX.name)
    cand_session, cand_input, cand_output = make_session_from_model(model_path)
    recs = parity_records.head(max_frames)

    panels = []
    for _, rec in recs.iterrows():
        bgr = imread_bgr(rec["image_path"])
        fp32_raw = run_raw_from_session(fp32_session, fp32_input, fp32_output, bgr)
        cand_raw = run_raw_from_session(cand_session, cand_input, cand_output, bgr)
        fp32_lanes = decode_raw_to_lanes(fp32_raw, decode_contract)
        cand_lanes = decode_raw_to_lanes(cand_raw, decode_contract)

        vis = bgr.copy()
        for lane in fp32_lanes:
            pts = np.asarray(lane["points"], dtype=np.int32)
            if len(pts) >= 2:
                cv2.polylines(vis, [pts], False, (0, 255, 0), 3)
        for lane in cand_lanes:
            pts = np.asarray(lane["points"], dtype=np.int32)
            if len(pts) >= 2:
                cv2.polylines(vis, [pts], False, (255, 0, 255), 1)
        cv2.putText(vis, f"{rec['set']} {rec['order']}", (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
        cv2.putText(vis, "green=fp32  magenta=quant", (20, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        panels.append(cv2.resize(vis, (648, 486)))

    if not panels:
        return None
    rows = []
    for i in range(0, len(panels), 2):
        row = panels[i:i+2]
        if len(row) == 1:
            row.append(np.zeros_like(row[0]))
        rows.append(np.hstack(row))
    sheet = np.vstack(rows)
    out = VIS_DIR / f"{selected_candidate['candidate']}_lane_compare.jpg"
    imwrite_bgr(out, sheet)
    print("wrote:", out)
    return out

visual_path = draw_lane_compare_sheet(selected)
visual_path

No selected candidate; skip visual sheet.


## 11b 결과를 읽는 법

결과 파일:

- `tables/onnx_node_inventory.csv`
  - ONNX node가 backbone/neck/heads 중 어디에 속하는지 확인하는 표

- `tables/candidate_eval_summary.csv`
  - 각 후보의 의미 보존과 latency 요약

- `reports/selective_quantization_boundary_report.json`
  - 최종 선택 후보와 다음 단계

판정:

- `meaning_pass=True`가 하나라도 있으면 그 후보만 Pi에서 다시 검증한다.
- `meaning_pass=False`만 있으면 아직 배포 후보가 아니다.
- local에서 빨라 보여도 Pi에서 빠르다는 보장은 없으므로, 최종 판정은 반드시 Pi latency를 봐야 한다.